# XGBoost — tercer modelo (ratio 1:1, siguiendo el protocolo de `tuning/v3`)

## Configuración del tuning
- **Dataset:** ratio **1:1** pseudo-ausencias:presencias — el ratio ganador identificado
  en el análisis de sensibilidad de [`tuning/v3`](../../tuning/v3/README.md), reconstruido
  aquí con la MISMA lógica (misma tabla base `pixel_year_full.csv`, mismo buffer de
  exclusión de 3 km, misma semilla `random_state=42`) para que sea comparable cifra por
  cifra con las filas `1:1` de LR y RF ya reportadas en `tuning/v3`.
- **Búsqueda de hiperparámetros:** `RandomizedSearchCV` con **150 configuraciones**
  muestreadas al azar (de 300 combinaciones posibles), `GroupKFold(10)` sobre bloques
  espaciales de 0.25°, `scoring='average_precision'`, **restringido a `year<=2019`**
  (misma corrección de fuga temporal que v2/v3).
- **`scale_pos_weight=1`** fijo (no se afina): con ratio 1:1 el dataset ya está
  balanceado (mismo número de presencias y pseudo-ausencias), así que no hace falta
  ponderar clases.

### Nota sobre la grilla pedida
La grilla original se pidió en términos de Random Forest (`n_estimators`, `max_depth`,
`max_features`, `min_samples_leaf`). XGBoost no tiene `max_features` ni
`min_samples_leaf` — se usan sus equivalentes nativos de boosting, preservando los
mismos valores donde es posible:

| Parámetro pedido (RF) | Valores pedidos | Equivalente en XGBoost | Valores usados |
|---|---|---|---|
| `n_estimators` | 100, 150, 200, 250, 350 | `n_estimators` (mismo nombre y valores) | 100, 150, 200, 250, 350 |
| `max_depth` | None, 10, 20, 30 | `max_depth` (0 = sin límite en XGBoost) | 0, 10, 20, 30 |
| `max_features` | 'sqrt', 0.5, 4 | `colsample_bytree` (fracción de columnas por árbol) | √9/9≈0.333, 0.5, 4/9≈0.444 |
| `min_samples_leaf` | 4, 6, 10, 20, 30 | `min_child_weight` (mínimo peso/muestras por hoja) | 4, 6, 10, 20, 30 |

$5 \times 4 \times 3 \times 5 = 300$ combinaciones posibles; se muestrean 150 al azar
(150 × 10 folds = 1,500 fits), una escala comparable a los 720 fits de RF en v1/v3.

## Qué se espera del modelo
XGBoost es un ensamble de árboles boosted secuenciales (cada árbol corrige los errores
del anterior), a diferencia de Random Forest (árboles independientes en paralelo). En
la literatura de susceptibilidad a incendios suele igualar o superar levemente a RF en
PR-AUC cuando el dataset es pequeño y las relaciones entre predictores son no lineales,
pero también es más propenso a sobreajustar con pocos datos (~4,154 filas aquí) si no
se regulariza bien — por eso se afinan `max_depth`, `min_child_weight` y
`colsample_bytree` en la búsqueda.

## Qué hace este notebook
1. Reconstruye el dataset ratio 1:1 desde `pixel_year_full.csv` (idéntico a `tuning/v3`).
2. Afina XGBoost con `RandomizedSearchCV` (150 configuraciones, `year<=2019`).
3. Evalúa el modelo afinado con el protocolo completo (spatial block CV 10 folds +
   hold-out temporal `>=2020`, con matrices de confusión) — igual que LR y RF.
4. Guarda los resultados en esta carpeta (`model/xgboost/`): `xgboost_metrics.csv` y
   `xgboost_vs_lr_rf_comparison.csv` (comparación directa contra LR y RF en ratio 1:1).

In [1]:
# === train_xgboost.ipynb — Setup: dataset ratio 1:1 (ratio ganador de tuning/v3) ===
# Reconstruye EXACTAMENTE el mismo dataset 1:1 que tuning/v3/tune_v3.ipynb generó para
# Random Forest y Logistic Regression (misma tabla base, mismo buffer de exclusión de
# 3 km, misma semilla random_state=42), para que XGBoost sea directamente comparable.
import numpy as np
import pandas as pd
from scipy.spatial import cKDTree

from xgboost import XGBClassifier
from sklearn.model_selection import RandomizedSearchCV, GroupKFold
from sklearn.metrics import (roc_auc_score, average_precision_score, f1_score,
                              confusion_matrix)
from sklearn.base import clone

pixel_year = pd.read_csv('../../data/model_dataset/pixel_year_full.csv')
pred_cols = ['dist_roads','dist_parks','dist_coca','dist_mosaic',
             'temp_C','vpd_kPa','ndvi','wind_ms','oni']
BLOCK = 0.25
EXCL_BUFFER = 3000            # 3 km — idéntico al usado en tuning/v3
buffer_deg = EXCL_BUFFER / 111000
RATIO = 1                     # ratio ganador identificado en tuning/v3 (ver tuning/v3/README.md)
SEED = 42

presences = pixel_year[pixel_year['burned'] == 1].copy()
absences_all = pixel_year[pixel_year['burned'] == 0].copy()

kept_absences = []
for y in sorted(pixel_year['year'].unique()):
    pres_y = presences[presences['year'] == y][['lon','lat']].values
    abs_y  = absences_all[absences_all['year'] == y]
    if len(pres_y) == 0:
        kept_absences.append(abs_y); continue
    tree = cKDTree(pres_y)
    dists, _ = tree.query(abs_y[['lon','lat']].values, k=1)
    kept_absences.append(abs_y[dists > buffer_deg])
absences_far = pd.concat(kept_absences, ignore_index=True)

n_abs = min(len(absences_far), int(round(RATIO * len(presences))))
absences_sample = absences_far.sample(n=n_abs, random_state=SEED)
model_df = pd.concat([presences, absences_sample], ignore_index=True) \
             .sample(frac=1, random_state=SEED).reset_index(drop=True)
model_df['block'] = (model_df['lon']//BLOCK).astype(int).astype(str) + '_' + \
                     (model_df['lat']//BLOCK).astype(int).astype(str)

n_pres = int(model_df['burned'].sum())
print("Dataset ratio 1:1 ->", model_df.shape,
      f"({n_pres} presencias, {model_df.shape[0]-n_pres} pseudo-ausencias)")

Dataset ratio 1:1 -> (4154, 14) (2077 presencias, 2077 pseudo-ausencias)


In [2]:
# === RandomizedSearchCV — 150 configuraciones aleatorias, tuning SOLO con year<=2019 ===
tune_df = model_df[model_df['year'] <= 2019].copy()
X_tune = tune_df[pred_cols].values
y_tune = tune_df['burned'].astype(int).values
groups_tune = tune_df['block'].values
cv = GroupKFold(n_splits=10)

param_distributions_xgb = {
    'n_estimators':     [100, 150, 200, 250, 350],
    'max_depth':        [0, 10, 20, 30],                 # 0 = sin límite (equivalente a None en RF)
    'colsample_bytree': [np.sqrt(len(pred_cols)) / len(pred_cols), 0.5, 4 / len(pred_cols)],
    'min_child_weight': [4, 6, 10, 20, 30],
}
n_combinations = (len(param_distributions_xgb['n_estimators']) *
                   len(param_distributions_xgb['max_depth']) *
                   len(param_distributions_xgb['colsample_bytree']) *
                   len(param_distributions_xgb['min_child_weight']))

xgb_base = XGBClassifier(
    objective='binary:logistic',
    eval_metric='aucpr',
    scale_pos_weight=1,     # dataset balanceado 1:1 -> sin corrección de clase
    tree_method='hist',
    random_state=42,
    n_jobs=-1,
)

xgb_search = RandomizedSearchCV(
    xgb_base,
    param_distributions=param_distributions_xgb,
    n_iter=150,
    scoring='average_precision',
    cv=cv,
    random_state=42,
    n_jobs=-1,
    refit=True,
)
xgb_search.fit(X_tune, y_tune, groups=groups_tune)

print(f"Grilla: {n_combinations} combinaciones posibles -> 150 muestreadas x 10 folds = "
      f"{150 * 10} fits")
print("Mejores hiperparámetros XGBoost:", xgb_search.best_params_)
print(f"Mejor PR-AUC (CV tuning, year<=2019): {xgb_search.best_score_:.3f}")

Grilla: 300 combinaciones posibles -> 150 muestreadas x 10 folds = 1500 fits
Mejores hiperparámetros XGBoost: {'n_estimators': 100, 'min_child_weight': 30, 'max_depth': 30, 'colsample_bytree': 0.4444444444444444}
Mejor PR-AUC (CV tuning, year<=2019): 0.867


In [3]:
def evaluate_model(name, estimator, model_df, pred_cols, n_splits=10, block=BLOCK):
    """Protocolo de validación IDÉNTICO a tuning/v3 (spatial block CV de 10 folds
    sobre el dataset completo + hold-out temporal train<=2019/test>=2020), con
    matrices de confusión — para que XGBoost sea directamente comparable con LR y RF."""
    X = model_df[pred_cols].values
    y = model_df['burned'].astype(int).values
    blocks = (model_df['lon']//block).astype(int).astype(str) + '_' + \
             (model_df['lat']//block).astype(int).astype(str)
    groups = blocks.values

    gkf = GroupKFold(n_splits=n_splits)
    rows = []
    cm_spatial = np.zeros((2, 2), dtype=int)
    for tr, te in gkf.split(X, y, groups):
        m = clone(estimator).fit(X[tr], y[tr])
        prob = m.predict_proba(X[te])[:, 1]
        pred = m.predict(X[te])
        auc = roc_auc_score(y[te], prob)
        prauc = average_precision_score(y[te], prob)
        f1 = f1_score(y[te], pred)
        rows.append((auc, prauc, f1))
        cm_spatial += confusion_matrix(y[te], pred, labels=[0, 1])
    r = np.array(rows)
    print(f"  [{name}] SPATIAL  AUC={r[:,0].mean():.3f}±{r[:,0].std():.3f}  "
          f"PR-AUC={r[:,1].mean():.3f}±{r[:,1].std():.3f}  F1={r[:,2].mean():.3f}±{r[:,2].std():.3f}")

    tr = (model_df['year'] <= 2019).values
    te = (model_df['year'] >= 2020).values
    m = clone(estimator).fit(X[tr], y[tr])
    prob = m.predict_proba(X[te])[:, 1]
    pred = m.predict(X[te])
    cm_temporal = confusion_matrix(y[te], pred, labels=[0, 1])
    auc_t = roc_auc_score(y[te], prob)
    prauc_t = average_precision_score(y[te], prob)
    f1_t = f1_score(y[te], pred)
    print(f"  [{name}] TEMPORAL AUC={auc_t:.3f}  PR-AUC={prauc_t:.3f}  F1={f1_t:.3f}")

    return {
        'spatial': r.mean(axis=0), 'spatial_std': r.std(axis=0), 'spatial_cm': cm_spatial,
        'temporal': (auc_t, prauc_t, f1_t), 'temporal_cm': cm_temporal,
    }


def cm_to_dict(cm, prefix):
    tn, fp, fn, tp = cm.ravel()
    return {f'{prefix}_tn': int(tn), f'{prefix}_fp': int(fp),
            f'{prefix}_fn': int(fn), f'{prefix}_tp': int(tp)}


res_xgb = evaluate_model("XGBoost (tuned, ratio 1:1)", xgb_search.best_estimator_, model_df, pred_cols)

  [XGBoost (tuned, ratio 1:1)] SPATIAL  AUC=0.871±0.027  PR-AUC=0.864±0.034  F1=0.789±0.046
  [XGBoost (tuned, ratio 1:1)] TEMPORAL AUC=0.811  PR-AUC=0.717  F1=0.693


In [4]:
# === Guardar resultados — mismo formato que tuning_v3_final_metrics.csv ===
row = {
    'ratio': '1:1',
    'model': 'XGBoost',
    'best_params': str(xgb_search.best_params_),
    'n_rows': model_df.shape[0],
    'n_presences': n_pres,
    'auc_spatial_mean': res_xgb['spatial'][0], 'auc_spatial_std': res_xgb['spatial_std'][0],
    'prauc_spatial_mean': res_xgb['spatial'][1], 'prauc_spatial_std': res_xgb['spatial_std'][1],
    'f1_spatial_mean': res_xgb['spatial'][2], 'f1_spatial_std': res_xgb['spatial_std'][2],
    'auc_temporal': res_xgb['temporal'][0], 'prauc_temporal': res_xgb['temporal'][1],
    'f1_temporal': res_xgb['temporal'][2],
}
row.update(cm_to_dict(res_xgb['spatial_cm'], 'cm_spatial'))
row.update(cm_to_dict(res_xgb['temporal_cm'], 'cm_temporal'))

xgb_metrics_df = pd.DataFrame([row])
xgb_metrics_df.to_csv('xgboost_metrics.csv', index=False)
print("Resultados guardados en: model/xgboost/xgboost_metrics.csv")
xgb_metrics_df[['ratio','model','n_rows','n_presences','auc_spatial_mean','prauc_spatial_mean',
                'f1_spatial_mean','auc_temporal','prauc_temporal','f1_temporal']]

Resultados guardados en: model/xgboost/xgboost_metrics.csv


,ratio,model,n_rows,n_presences,auc_spatial_mean,prauc_spatial_mean,f1_spatial_mean,auc_temporal,prauc_temporal,f1_temporal
0,1:1,XGBoost,4154,2077,0.87108,0.86377,0.788643,0.810905,0.716539,0.69258


In [5]:
# === Comparación directa contra LR y RF (mismo ratio 1:1, mismo protocolo de validación) ===
v3_sensitivity = pd.read_csv('../../tuning/v3/tuning_v3_sensitivity_metrics.csv')
cols = ['ratio','model','n_rows','n_presences','auc_spatial_mean','prauc_spatial_mean',
        'f1_spatial_mean','auc_temporal','prauc_temporal','f1_temporal']
lr_rf_11 = v3_sensitivity[v3_sensitivity['ratio'] == '1:1'][cols]

comparison_df = pd.concat([lr_rf_11, xgb_metrics_df[cols]], ignore_index=True)
comparison_df.to_csv('xgboost_vs_lr_rf_comparison.csv', index=False)
print("Comparación guardada en: model/xgboost/xgboost_vs_lr_rf_comparison.csv\n")
comparison_df

Comparación guardada en: model/xgboost/xgboost_vs_lr_rf_comparison.csv



,ratio,model,n_rows,n_presences,auc_spatial_mean,prauc_spatial_mean,f1_spatial_mean,auc_temporal,prauc_temporal,f1_temporal
0,1:1,Logistic Regression,4154,2077,0.845518,0.822546,0.753945,0.809537,0.746931,0.663230
1,1:1,Random Forest,4154,2077,0.878350,0.865608,0.789538,0.817119,0.703800,0.684303
2,1:1,XGBoost,4154,2077,0.871080,0.863770,0.788643,0.810905,0.716539,0.692580


## Conclusiones

**Mejores hiperparámetros (`RandomizedSearchCV`, 150/300 combinaciones, tuning `year<=2019`):**
`n_estimators=100`, `max_depth=30`, `colsample_bytree=0.444` (equivalente a `max_features=4`),
`min_child_weight=30`. PR-AUC de tuning (CV espacial, `year<=2019`): **0.867**.

### Resultados (ratio 1:1, dataset idéntico al de LR/RF en `tuning/v3`)

| Modelo | AUC espacial | PR-AUC espacial | F1 espacial | AUC temporal | PR-AUC temporal | F1 temporal |
|---|---|---|---|---|---|---|
| Logistic Regression | 0.846 | 0.823 | 0.754 | 0.810 | 0.747 | 0.663 |
| Random Forest | 0.878 | **0.866** | 0.790 | 0.817 | 0.704 | 0.684 |
| **XGBoost** | 0.871 | 0.864 | 0.789 | 0.811 | **0.717** | **0.693** |

(tabla completa en [`xgboost_vs_lr_rf_comparison.csv`](xgboost_vs_lr_rf_comparison.csv))

### ¿XGBoost mejora la predicción?

- **Validación espacial:** XGBoost prácticamente empata con Random Forest (PR-AUC
  0.864 vs. 0.866, diferencia de 0.002 — muy por debajo de la desviación estándar
  entre folds de ambos modelos, ~0.03-0.04). No hay una mejora real sobre RF al
  generalizar a **lugares** nuevos.
- **Validación temporal (train ≤2019 / test ≥2020):** XGBoost sí mejora sobre RF en
  PR-AUC (0.717 vs. 0.704, **+0.013**) y en F1 (0.693 vs. 0.684, **+0.009**), aunque
  AUC-ROC es levemente menor (0.811 vs. 0.817). Como PR-AUC es la métrica de
  selección del proyecto (evento raro), esto indica que XGBoost generaliza mejor a
  **años** nuevos que Random Forest, con el mismo dataset y protocolo de validación.
- Frente a Logistic Regression, XGBoost mejora ambas validaciones con margen amplio
  (PR-AUC espacial +0.041, PR-AUC temporal +0.030), confirmando que el modelo no
  lineal sigue aportando valor sobre la línea base.

### Conclusión para el mapa de susceptibilidad 2026

Dado que el mapa 2026 es una **proyección a un año futuro no observado** (una
extrapolación temporal, no espacial), la métrica más relevante es el hold-out
temporal, no la espacial. Ahí, **XGBoost obtiene el mejor PR-AUC y F1 de los tres
modelos**, superando a Random Forest en la validación que más se parece a la tarea
real de predecir 2026. La mejora es modesta (+0.013 PR-AUC) y no cambia la conclusión
del proyecto de que el ratio de muestreo (1:1 vs. 2:1, ver `tuning/v3`) importa mucho
más que la elección de algoritmo — pero sí sugiere que **XGBoost, entrenado con el
dataset 1:1, es la mejor opción disponible para generar el mapa de susceptibilidad
2026**, con Random Forest como alternativa muy cercana y más simple de interpretar.